In [1]:
import requests

URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"


def ask(messages):
    response = requests.post(
        URL,
        json={
            "model": MODEL,
            "messages": messages,
            "stream": False,
            "options": {"num_predict": 128},
        },
        timeout=180,
    )
    response.raise_for_status()
    return response.json()


messages = [
    {
        "role": "system",
        "content": "Réponds brièvement dans la langue de l'utilisateur.",
    },
    {"role": "user", "content": "Je m'appelle Lina."},
]

result = ask(messages)
print(result["message"]["content"])

# TODO : ajouter la réponse de l'assistant à messages.
messages.append(result['message'])
# TODO : ajouter une question utilisateur : « Comment je m'appelle ? »
messages.append({"role": "user", "content": "Comment je m'appelle ?"})
# TODO : appeler à nouveau ask et afficher la réponse.
result = ask(messages)
messages.append(result['message'])
print(result["message"]["content"])
# TODO : poursuivre avec « Réponds maintenant en anglais. »
messages.append({"role": "user", "content": "Réponds maintenant en anglais"})
result = ask(messages)
messages.append(result['message'])
print(result["message"]["content"])


Bonjour Lina, à quoi puis-je vous aider aujourd'hui?
Je m'appelle Lina.
You asked "How should I respond that I am Lina?". The answer in English would be: "That's how you should respond that I am Lina."


In [ ]:
import requests

URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"


def ask(messages):
    response = requests.post(
        URL,
        json={
            "model": MODEL,
            "messages": messages,
            "stream": False,
            "options": {"num_predict": 128},
        },
        timeout=180,
    )
    response.raise_for_status()
    return response.json()


messages = [
    {
        "role": "system",
        "content": "Réponds brièvement dans la langue de l'utilisateur.",
    },
    {"role": "user", "content": "Je m'appelle Lina."},
]

result = ask(messages)
print(result["message"]["content"])

# TODO : ajouter la réponse de l'assistant à messages.
messages.append(result['message'])
# TODO : ajouter une question utilisateur : « Comment je m'appelle ? »
messages.append({"role": "user", "content": "Comment je m'appelle ?"})
# TODO : appeler à nouveau ask et afficher la réponse.
result = ask(messages)
messages.append(result['message'])
print(result["message"]["content"])
# TODO : poursuivre avec « Réponds maintenant en anglais. »
messages.append({"role": "user", "content": "Réponds maintenant en anglais"})
result = ask(messages)
messages.append(result['message'])
print(result["message"]["content"])


{'role': 'assistant', 'content': "Bonjour Lina, à votre service. Comment puis-je vous aider aujourd'hui ?"}


Streaming


In [ ]:
import json
from time import perf_counter

import requests


def ask_stream(messages):
    start = perf_counter()
    first_content_at = None
    parts = []

    with requests.post(
        URL,
        json={
            "model": MODEL,
            "messages": messages,
            "stream": True,
            "keep_alive": "10m",
            "options": {
                "num_ctx": 512,
                "num_predict": 64},
            },
        stream=True,
        timeout=180,
    ) as response:
        response.raise_for_status()

        for line in response.iter_lines():
            if not line:
                continue

            event = json.loads(line)
            if "error" in event:
                raise RuntimeError(event["error"])

            content = event.get("message", {}).get("content", "")

            if content:
                # TODO 1 : enregistrer l'instant du premier contenu,
                # uniquement si first_content_at vaut encore None.
                if  first_content_at is None:
                    first_content_at = perf_counter()
                # TODO 2 : conserver ce fragment dans parts.
                parts.append(content)
                print(content, end="", flush=True)

            if event.get("done"):
                break

    end = perf_counter()
    
    print()
    answer = ''.join(parts)
    first_content_delay = (first_content_at - start 
                           if first_content_at is not None
                           else None)
    total_time = (end - start)

    return answer, first_content_delay, total_time

answer, delay, duration = ask_stream(messages)
print("Réponse reconstruite :", answer)
print("Délai avant premier texte :", delay, "secondes")
print("Durée totale :", duration, "secondes")
messages.append({"role":"assistant", "content": answer})


Réponse reconstruite : 
Délai avant premier texte : None secondes
Durée totale : 12.547773699974641 secondes


Context window et RAM usage

In [6]:
import json
from time import perf_counter

import requests


def ask_stream(messages, num_ctx=512):
    start = perf_counter()
    first_content_at = None
    parts = []

    with requests.post(
        URL,
        json={
            "model": MODEL,
            "messages": messages,
            "stream": True,
            "keep_alive": "10m",
            "options": {
                "num_ctx": num_ctx,
                "num_predict": 64,
                "temperature": 0
                },
            },
        stream=True,
        timeout=180,
    ) as response:
        response.raise_for_status()

        for line in response.iter_lines():
            if not line:
                continue

            event = json.loads(line)
            if "error" in event:
                raise RuntimeError(event["error"])

            content = event.get("message", {}).get("content", "")

            if content:
                # TODO 1 : enregistrer l'instant du premier contenu,
                # uniquement si first_content_at vaut encore None.
                if  first_content_at is None:
                    first_content_at = perf_counter()
                # TODO 2 : conserver ce fragment dans parts.
                parts.append(content)
                print(content, end="", flush=True)

            if event.get("done"):
                print("\n--- Mesures serveur ---")

                # Ces durées sont fournies en nanosecondes.
                print("Chargement :", event["load_duration"] / 1e9, "s")
                print("Traitement entrée :", event["prompt_eval_duration"] / 1e9, "s")
                print("Génération :", event["eval_duration"] / 1e9, "s")

                print("Tokens d'entrée traités :", event["prompt_eval_count"])
                print("Tokens générés :", event["eval_count"])
                break

    end = perf_counter()
    
    print()
    answer = ''.join(parts)
    first_content_delay = (first_content_at - start 
                           if first_content_at is not None
                           else None)
    total_time = (end - start)

    return answer, first_content_delay, total_time
test_messages = [
    {"role": "system", "content": "Réponds en français"},
    {"role": "user", "content": "Explique moi en deux phrases ce qu'est un CPU"}
]

for attempt in range(2):
    print(f"\nEssai {attempt + 1}")
    answer, delay, duration = ask_stream(test_messages, num_ctx=2048)
    print("Premier texte :", delay, "| Total :", duration)


Essai 1
Le CPU, ou Central Processing Unit, est le cœur du processeur d'un ordinateur. Il gère et effectue toutes les opérations de traitement de données, exécutant les instructions du logiciel et des programmes.
--- Mesures serveur ---
Chargement : 13.3927088 s
Traitement entrée : 1.217213 s
Génération : 8.038805 s
Tokens d'entrée traités : 29
Tokens générés : 51

Premier texte : 16.671428099973127 | Total : 24.705538999987766

Essai 2
Le CPU, ou Central Processing Unit, est le cœur du processeur d'un ordinateur. Il gère et effectue toutes les opérations de traitement de données, exécutant les instructions du logiciel et des programmes.
--- Mesures serveur ---
Chargement : 0.0043536 s
Traitement entrée : 0.129217 s
Génération : 6.224668 s
Tokens d'entrée traités : 29
Tokens générés : 51

Premier texte : 2.1784037000034004 | Total : 8.40374730003532


Multiple conversations

In [ ]:
histories = {}


def get_history(user_id, conversation_id):
    """
    Renvoie la liste des messages de cette conversation.

    Si elle n'existe pas :
    - crée une nouvelle liste contenant un message système ;
    - conserve cette liste dans histories.

    Si elle existe :
    - renvoie l'historique déjà enregistré, sans le réinitialiser.
    """
    # À compléter